# Variational Quantum Circuit

PCA features
    │
    ▼
Angle Encoding
    │
    ▼
VQC
    │
    ├── Rot gates
    ├── CNOT entanglement
    └── N layers
    │
    ▼
Measure ALL qubits
    │
    ▼
n_qubits expectation values
    │
    ▼
Classical trainable layer
    │
    ▼
4 logits
    │
    ▼
Softmax
    │
    ▼
4 classes

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
import seaborn as sns

import tensorly as tl
from tensorly.decomposition import tensor_train
from tensorly.tt_tensor import tt_to_tensor
from pennylane import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_recall_fscore_support

import pennylane as qml
from pennylane import numpy as pnp

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

## Experiment 2: New Architecture

In [5]:
# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent.parent

ANGLE_ROOT = Path(PROJECT_ROOT/"data/vqc/angle_encoding")

RESULTS_ROOT = Path(PROJECT_ROOT/"results/vqc/experiment4")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

EXCEL_PATH = (
    RESULTS_ROOT /
    "vqc_pca_qubit_comparison.xlsx"
)

# ============================================================
# SETTINGS
# ============================================================

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

QUBIT_LIST = [
    4,
    8,
    12,
    16
]

N_LAYERS = 2
N_CLASSES = 4
N_EPOCHS = 30
LEARNING_RATE = 0.05
BATCH_SIZE = 32
RANDOM_SEED = 42

TASK_NAMES = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

# RANDOM SEED
np.random.seed(
    RANDOM_SEED
)

CLASS_IDS = list(
    range(N_CLASSES)
)

# ============================================================
# FUNCTIONS
# ============================================================

def softmax(x):

    x = np.asarray(x)

    x = x - pnp.max(
        x,
        axis=-1,
        keepdims=True
    )

    exp_x = np.exp(x)

    return (
        exp_x
        /
        pnp.sum(
            exp_x,
            axis=-1,
            keepdims=True
        )
    )

def initialise_weights(n_qubits, seed):

    rng = np.random.default_rng(seed)

    weights = (
        0.01
        * rng.standard_normal(
            (
                N_LAYERS,
                n_qubits,
                3
            )
        )
    )

    return pnp.array(
        weights,
        requires_grad=True
    )

def initialise_classical_layer(n_qubits, seed):

    rng = np.random.default_rng(
        seed
    )

    W = (
        0.01
        * np.random.randn(
            n_qubits,
            N_CLASSES
        )
    )

    b = np.zeros(
        N_CLASSES
    )

    return (
        pnp.array(
            W,
            requires_grad=True
        ),
        pnp.array(
            b,
            requires_grad=True
        )
    )


def create_quantum_circuit(n_qubits):

    dev = qml.device(
        "default.qubit",
        wires=n_qubits
    )

    @qml.qnode(dev)
    def quantum_circuit(
        x,
        weights
    ):

        # ANGLE ENCODING
        qml.AngleEmbedding(
            x,
            wires=range(n_qubits),
            rotation="Y"
        )

        for layer in range(N_LAYERS):

            # TRAINABLE ROTATIONS
            for qubit in range(n_qubits):

                # Each Rot gate has three trainable parameters.
                # So 2 layers × 8 qubits × 3 = 48 trainable parameters.
                qml.Rot(
                    weights[layer, qubit, 0],
                    weights[layer, qubit, 1],
                    weights[layer, qubit, 2],
                    wires=qubit
                )

            # ENTANGLEMENT
            for qubit in range(
                n_qubits - 1
            ):

                qml.CNOT(
                    wires=[
                        qubit,
                        qubit + 1
                    ]
                )

        # MEASUREMENTS
        return [
            qml.expval(
                qml.PauliZ(qubit)
            )
            for qubit in range(n_qubits)
        ]

    return quantum_circuit

# FORWARD PASS

def predict_logits(
    X,
    quantum_weights,
    classical_weights,
    classical_bias,
    quantum_circuit
):

    outputs = []

    for sample in X:

        quantum_output = quantum_circuit(
            sample,
            quantum_weights
        )

        quantum_output = pnp.asarray(
            quantum_output
        )

        logits = (
            pnp.dot(
                quantum_output,
                classical_weights
            )
            +
            classical_bias
        )

        outputs.append(
            logits
        )

    return pnp.stack(
        outputs
    )


# CROSS-ENTROPY

def cross_entropy(
    probabilities,
    labels
):

    probabilities = pnp.clip(
        probabilities,
        1e-10,
        1.0
    )

    selected_probabilities = (
        probabilities[
            pnp.arange(
                len(labels)
            ),
            labels
        ]
    )

    losses = -pnp.log(
        selected_probabilities
    )

    return pnp.mean(
        losses
    )


def prediction_distribution(predictions):

    counts = np.bincount(
        predictions,
        minlength=N_CLASSES
    )

    percentages = (
        counts
        /
        len(predictions)
        *
        100
    )

    return counts, percentages

# ============================================================
# TRAIN VQC
# ============================================================

def train_vqc(
    X_train,
    y_train,
    n_qubits,
    seed
):

    quantum_circuit = (
        create_quantum_circuit(
            n_qubits
        )
    )

    quantum_weights = (
        initialise_weights(
            n_qubits,
            seed
        )
    )

    classical_weights, classical_bias = (
        initialise_classical_layer(
            n_qubits,
            seed + 1000
        )
    )

    # OPTIMISER
    opt = qml.AdamOptimizer(
        stepsize=LEARNING_RATE
    )

    # COST FUNCTION
    def cost_fn(
        quantum_weights,
        classical_weights,
        classical_bias,
        X_batch,
        y_batch
    ):

        logits = predict_logits(
            X_batch,
            quantum_weights,
            classical_weights,
            classical_bias,
            quantum_circuit
        )

        probabilities = softmax(
            logits
        )

        return cross_entropy(
            probabilities,
            y_batch
        )

    # TRAINING LOOP

    loss_history = []
    accuracy_history = []

    for epoch in range(
        N_EPOCHS
    ):

        # Random mini-batch
        batch_size = min(
            BATCH_SIZE,
            len(X_train)
        )

        batch_indices = np.random.choice(
            len(X_train),
            size=batch_size,
            replace=False
        )

        X_batch = X_train[
            batch_indices
        ]

        y_batch = y_train[
            batch_indices
        ]

        (
            quantum_weights,
            classical_weights,
            classical_bias
        ), loss = opt.step_and_cost(

            lambda qw, cw, cb:
                cost_fn(
                    qw,
                    cw,
                    cb,
                    X_batch,
                    y_batch
                ),

            quantum_weights,
            classical_weights,
            classical_bias
        )

        loss_value = float(
            loss
        )

        loss_history.append(
            loss_value
        )

        batch_logits = predict_logits(
            X_batch,
            quantum_weights,
            classical_weights,
            classical_bias,
            quantum_circuit
        )

        batch_predictions = np.argmax(
            np.asarray(
                batch_logits
            ),
            axis=1
        )

        batch_accuracy = accuracy_score(
            y_batch,
            batch_predictions
        )

        accuracy_history.append(
            batch_accuracy
        )

        if (
            epoch == 0
            or
            (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch "
                f"{epoch + 1:3d}/{N_EPOCHS} "
                f"| Loss = "
                f"{loss:.6f}"
            )

    return (
        quantum_weights,
        classical_weights,
        classical_bias,
        quantum_circuit,
        loss_history,
        accuracy_history
    )


# ============================================================
# PREDICTION
# ============================================================

def predict(
    X,
    quantum_weights,
    classical_weights,
    classical_bias,
    quantum_circuit
):

    logits = predict_logits(
        X,
        quantum_weights,
        classical_weights,
        classical_bias,
        quantum_circuit
    )

    probabilities = softmax(
        logits
    )

    probabilities_np = np.asarray(
        probabilities
    )

    predictions = np.argmax(
        probabilities_np,
        axis=1
    )

    return (
        predictions,
        probabilities_np
    )

all_results = []
task_results = []

loss_histories = {
    n_qubits: []
    for n_qubits in QUBIT_LIST
}

accuracy_histories = {
    n_qubits: []
    for n_qubits in QUBIT_LIST
}

confusion_matrices = {
    n_qubits: np.zeros(
        (
            N_CLASSES,
            N_CLASSES
        ),
        dtype=int
    )
    for n_qubits in QUBIT_LIST
}

task_counts = []

for n_qubits in QUBIT_LIST:

    print("\n")
    print("=" * 80)
    print(
        f"VQC EXPERIMENT: "
        f"{n_qubits} QUBITS"
    )
    print("=" * 80)

    # Check angle encoding directory

    for subject_index, test_subject in enumerate(
        SUBJECTS
    ):

        print("\n")
        print("-" * 80)

        print(
            f"QUBITS = {n_qubits}"
        )

        print(
            f"TEST SUBJECT = "
            f"{test_subject}"
        )

        print("-" * 80)

        # Load corresponding angle data

        angle_path = (
            ANGLE_ROOT
            /
            f"loso_test_{test_subject}"
            /
            f"angle_{n_qubits}"
            /
            "data.npz"
        )

        if not angle_path.exists():

            raise FileNotFoundError(
                f"\nMissing angle file:\n"
                f"{angle_path}\n\n"
                f"Make sure PCA/angle encoding "
                f"has been generated for "
                f"{n_qubits} components."
            )

        data = np.load(
            angle_path,
            allow_pickle=True
        )

        X_train = data[
            "angles_train"
        ]

        X_test = data[
            "angles_test"
        ]

        y_train = np.asarray(
            data["y_train"],
            dtype=int
        )

        y_test = np.asarray(
            data["y_test"],
            dtype=int
        )

        # Check dimensions

        if X_train.shape[1] != n_qubits:

            raise ValueError(
                f"Expected {n_qubits} "
                f"features but received "
                f"{X_train.shape[1]}"
            )

        print(
            "Training:",
            X_train.shape
        )

        print(
            "Testing:",
            X_test.shape
        )

        print(
            "Training labels:",
            np.unique(y_train)
        )

        print(
            "Testing labels:",
            np.unique(y_test)
        )


        for class_id, task_name in enumerate(
            TASK_NAMES
        ):

            count = np.sum(
                y_train == class_id
            )

            task_counts.append({

                "subject":
                    test_subject,

                "n_qubits":
                    n_qubits,

                "task":
                    task_name,

                "count":
                    int(count)
            })

        # TRAIN
        
        print("\n")
        print(
            "Training VQC..."
        )

        seed = (
            RANDOM_SEED
            +
            n_qubits * 100
            +
            subject_index
        )

        (
            quantum_weights,
            classical_weights,
            classical_bias,
            quantum_circuit,
            loss_history,
            accuracy_history
        ) = train_vqc(
            X_train,
            y_train,
            n_qubits,
            seed
        )

        loss_histories[
            n_qubits
        ].append(
            loss_history
        )

        accuracy_histories[
            n_qubits
        ].append(
            accuracy_history
        )

        # TEST

        print("\n")
        print(
            "Testing VQC..."
        )

        predictions, probabilities = (
            predict(
                X_test,
                quantum_weights,
                classical_weights,
                classical_bias,
                quantum_circuit
            )
        )

        prediction_counts, prediction_percentages = (
            prediction_distribution(
                predictions
            )
        )

        prediction_collapse = np.max(
            prediction_percentages
        )

        # METRICS

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        balanced_accuracy = (
            balanced_accuracy_score(
                y_test,
                predictions
            )
        )

        macro_f1 = f1_score(
            y_test,
            predictions,
            labels=CLASS_IDS,
            average="macro",
            zero_division=0
        )

        weighted_f1 = f1_score(
            y_test,
            predictions,
            labels=CLASS_IDS,
            average="weighted",
            zero_division=0
        )

        macro_precision = precision_score(
            y_test,
            predictions,
            labels=CLASS_IDS,
            average="macro",
            zero_division=0
        )

        macro_recall = recall_score(
            y_test,
            predictions,
            labels=CLASS_IDS,
            average="macro",
            zero_division=0
        )

        cm = confusion_matrix(
            y_test,
            predictions,
            labels=CLASS_IDS
        )

        confusion_matrices[
            n_qubits
        ] += cm

        print("\nPrediction distribution:")

        for class_id in range(N_CLASSES):

            print(
                f"Class {class_id}: "
                f"{prediction_counts[class_id]} "
                f"("
                f"{prediction_percentages[class_id]:.2f}%"
                f")"
            )

        print(
            "\nTest accuracy:",
            f"{accuracy:.4f}"
        )

        print(
            "Balanced accuracy:",
            f"{balanced_accuracy:.4f}"
        )

        print(
            "Macro F1:",
            f"{macro_f1:.4f}"
        )

        print(
            "Weighted F1:",
            f"{weighted_f1:.4f}"
        )

        print(
            "\nConfusion matrix:"
        )

        print(cm)

        for class_id, task_name in enumerate(
            TASK_NAMES
        ):

            y_true_binary = (
                y_test == class_id
            )

            y_pred_binary = (
                predictions == class_id
            )

            task_precision = precision_score(
                y_true_binary,
                y_pred_binary,
                zero_division=0
            )

            task_recall = recall_score(
                y_true_binary,
                y_pred_binary,
                zero_division=0
            )

            task_f1 = f1_score(
                y_true_binary,
                y_pred_binary,
                zero_division=0
            )

            task_results.append({

                "subject":
                    test_subject,

                "n_qubits":
                    n_qubits,

                "task":
                    task_name,

                "precision":
                    task_precision,

                "recall":
                    task_recall,

                "f1":
                    task_f1,

                "support":
                    int(
                        np.sum(
                            y_test == class_id
                        )
                    )
            })

        all_results.append({

            "subject":
                test_subject,

            "n_qubits":
                n_qubits,

            "accuracy":
                accuracy,

            "balanced_accuracy":
                balanced_accuracy,

            "macro_precision":
                macro_precision,

            "macro_recall":
                macro_recall,

            "macro_f1":
                macro_f1,

            "weighted_f1":
                weighted_f1,

            "final_training_loss":
                loss_history[-1],

            "final_training_accuracy":
                accuracy_history[-1],

            "prediction_collapse":
                prediction_collapse
        })

        # SAVE INDIVIDUAL RESULT

        result_dir = (
            RESULTS_ROOT
            /
            f"qubits_{n_qubits}"
        )

        result_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        save_path = (
            result_dir
            /
            f"loso_test_{test_subject}.npz"
        )

        np.savez_compressed(

            save_path,
            test_subject=test_subject,
            predictions=predictions,
            prediction_collapse=prediction_collapse,
            probabilities=probabilities,
            y_test=y_test,
            accuracy=accuracy,
            balanced_accuracy=balanced_accuracy,
            macro_precision=macro_precision,
            macro_recall=macro_recall,
            macro_f1=macro_f1,
            weighted_f1=weighted_f1,
            confusion_matrix=cm,
            quantum_weights=np.asarray(
                quantum_weights
            ),
            classical_weights=np.asarray(
                classical_weights
            ),
            classical_bias=np.asarray(
                classical_bias
            ),
            loss_history=np.asarray(
                loss_history
            ),
            accuracy_history=np.asarray(
                accuracy_history
            ),
            n_qubits=n_qubits,
            n_layers=N_LAYERS
        )

        print(
            "Saved:",
            save_path
        )

        # STORE ROW FOR EXCEL

        all_results.append({

            "subject": test_subject,
            "n_qubits": n_qubits,
            "accuracy": accuracy,
            "balanced_accuracy":
                balanced_accuracy,
            "macro_f1":
                macro_f1,
            "weighted_f1":
                weighted_f1,
            "final_training_loss":
                loss_history[-1],
            "prediction_collapse":
                prediction_collapse
        })

# CONVERT RESULTS TO DATAFRAME

results_df = pd.DataFrame(
    all_results
)

task_results_df = pd.DataFrame(
    task_results
)

task_counts_df = pd.DataFrame(
    task_counts
)

# LOSO SUMMARY

summary_df = (
    results_df
    .groupby(
        "n_qubits"
    )
    .agg({
        "accuracy":
            ["mean", "std"],

        "balanced_accuracy":
            ["mean", "std"],

        "macro_precision":
            ["mean", "std"],

        "macro_recall":
            ["mean", "std"],

        "macro_f1":
            ["mean", "std"],

        "weighted_f1":
            ["mean", "std"],

        "final_training_loss":
            ["mean", "std"],

        "final_training_accuracy":
            ["mean", "std"],

        "prediction_collapse":
            ["mean", "std"]
    })
    .reset_index()
)

# Flatten column names

summary_df.columns = [

    "n_qubits",

    "accuracy_mean",
    "accuracy_std",

    "balanced_accuracy_mean",
    "balanced_accuracy_std",

    "macro_precision_mean",
    "macro_precision_std",

    "macro_recall_mean",
    "macro_recall_std",

    "macro_f1_mean",
    "macro_f1_std",

    "weighted_f1_mean",
    "weighted_f1_std",

    "loss_mean",
    "loss_std",

    "training_accuracy_mean",
    "training_accuracy_std",

    "prediction_collapse_mean",
    "prediction_collapse_std"
]

# TASK-WISE SUMMARY
# ============================================================

task_summary_df = (
    task_results_df
    .groupby(
        ["task", "n_qubits"]
    )
    .agg({

        "precision":
            ["mean", "std"],

        "recall":
            ["mean", "std"],

        "f1":
            ["mean", "std"],

        "support":
            ["mean", "std"]
    })
    .reset_index()
)

task_summary_df.columns = [

    "task",
    "n_qubits",

    "precision_mean",
    "precision_std",

    "recall_mean",
    "recall_std",

    "f1_mean",
    "f1_std",

    "support_mean",
    "support_std"
]

# SAVE EXCEL

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="LOSO Results",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    task_results_df.to_excel(
        writer,
        sheet_name="Task Results",
        index=False
    )

    task_summary_df.to_excel(
        writer,
        sheet_name="Task Summary",
        index=False
    )

    task_counts_df.to_excel(
        writer,
        sheet_name="Training Counts",
        index=False
    )


print("\n")
print("=" * 80)
print("EXCEL RESULTS SAVED")
print("=" * 80)

print(
    EXCEL_PATH
)

# PRINT SUMMARY

print("\n")
print("=" * 80)
print("VQC QUANTUM DIMENSION SUMMARY")
print("=" * 80)

print(
    summary_df.to_string(
        index=False
    )
)


# ============================================================
# PLOT 1:
# MEAN ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["accuracy_mean"],
    yerr=summary_df["accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "VQC Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


accuracy_plot = (
    RESULTS_ROOT /
    "accuracy_vs_qubits.png"
)

plt.savefig(
    accuracy_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 2:
# MACRO F1 VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["macro_f1_mean"],
    yerr=summary_df["macro_f1_std"],
    marker="o",
    capsize=5
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Macro F1"
)

plt.title(
    "VQC Macro F1 vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()


f1_plot = (
    RESULTS_ROOT /
    "macro_f1_vs_qubits.png"
)

plt.savefig(
    f1_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 3:
# BALANCED ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["balanced_accuracy_mean"],
    yerr=summary_df["balanced_accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Balanced accuracy"
)

plt.title(
    "VQC Balanced Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


balanced_plot = (
    RESULTS_ROOT /
    "balanced_accuracy_vs_qubits.png"
)

plt.savefig(
    balanced_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 4:
# ACCURACY FOR EACH SUBJECT
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = results_df[
        results_df["subject"] == subject
    ]

    plt.plot(
        subject_data["n_qubits"],
        subject_data["accuracy"],
        marker="o",
        label=f"Subject {subject}"
    )


plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "LOSO Accuracy Across Quantum Dimensions"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


subject_plot = (
    RESULTS_ROOT /
    "accuracy_by_subject.png"
)

plt.savefig(
    subject_plot,
    dpi=300
)

plt.close()

# ============================================================
# PLOT 5:
# NUMBER OF TRAINING SAMPLES PER TASK
# ============================================================

task_counts = []

for test_subject in SUBJECTS:

    for n_qubits in QUBIT_LIST:

        angle_path = (
            ANGLE_ROOT
            / f"loso_test_{test_subject}"
            / f"angle_{n_qubits}"
            / "data.npz"
        )

        data = np.load(
            angle_path,
            allow_pickle=True
        )

        y_train = data["y_train"]

        for class_id, task_name in enumerate(TASK_NAMES):

            count = np.sum(
                y_train == class_id
            )

            task_counts.append({
                "subject": test_subject,
                "n_qubits": n_qubits,
                "task": task_name,
                "count": count
            })

task_counts_df = pd.DataFrame(
    task_counts
)

distribution_df = (
    task_counts_df[
        task_counts_df["n_qubits"] == 4
    ]
)

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = distribution_df[
        distribution_df["subject"] == subject
    ]

    plt.plot(
        subject_data["task"],
        subject_data["count"],
        marker="o",
        label=f"Subject {subject}"
    )

plt.xlabel("Task")
plt.ylabel("Number of training trials")
plt.title(
    "Training Trial Distribution Across Tasks"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

task_distribution_plot = (
    RESULTS_ROOT /
    "training_samples_by_task.png"
)

plt.savefig(
    task_distribution_plot,
    dpi=300
)

plt.close()

# ============================================================
# PLOT 6:
# Training Data Composition Across Tasks
# ============================================================

task_percentage = (
    distribution_df
    .groupby(["subject", "task"])["count"]
    .sum()
    .reset_index()
)

task_percentage["percentage"] = (
    task_percentage["count"]
    /
    task_percentage.groupby("subject")["count"]
        .transform("sum")
    * 100
)

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = task_percentage[
        task_percentage["subject"] == subject
    ]

    plt.plot(
        subject_data["task"],
        subject_data["percentage"],
        marker="o",
        label=f"Subject {subject}"
    )

plt.xlabel("Task")
plt.ylabel("Training samples (%)")
plt.title(
    "Training Data Composition Across Tasks"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "training_task_percentage.png",
    dpi=300
)

plt.close()

# ============================================================
# PLOT 7:
# Task-wise F1 Across Quantum Dimensions
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for task in TASK_NAMES:

    task_data = (
        task_results_df[
            task_results_df["task"] == task
        ]
        .groupby("n_qubits")["f1"]
        .mean()
        .reset_index()
    )

    plt.plot(
        task_data["n_qubits"],
        task_data["f1"],
        marker="o",
        label=task
    )

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Mean task F1"
)

plt.title(
    "Task-wise F1 Across Quantum Dimensions"
)

plt.xticks(QUBIT_LIST)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "task_f1_vs_qubits.png",
    dpi=300
)

plt.close()

# ==========================================================
# Task-wise VQC Recall Across Quantum Dimensions
# ==========================================================

plt.figure(
    figsize=(9, 6)
)

for task in TASK_NAMES:

    task_data = (
        task_results_df[
            task_results_df["task"] == task
        ]
        .groupby("n_qubits")["recall"]
        .mean()
        .reset_index()
    )

    plt.plot(
        task_data["n_qubits"],
        task_data["recall"],
        marker="o",
        label=task
    )

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Mean task recall"
)

plt.title(
    "Task-wise VQC Recall Across Quantum Dimensions"
)

plt.xticks(QUBIT_LIST)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "task_recall_vs_qubits.png",
    dpi=300
)

plt.close()

# ============================================================
# TASK PRECISION
# ============================================================

plt.figure(figsize=(9, 6))

for task in TASK_NAMES:

    task_data = (
        task_results_df[
            task_results_df["task"] == task
        ]
        .groupby("n_qubits")["precision"]
        .mean()
        .reset_index()
    )

    plt.plot(
        task_data["n_qubits"],
        task_data["precision"],
        marker="o",
        label=task
    )

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Mean task precision"
)

plt.title(
    "Task-wise Precision Across Quantum Dimensions"
)

plt.xticks(QUBIT_LIST)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "task_precision_vs_qubits.png",
    dpi=300
)

plt.close()

# ============================================================
# TRAINING LOSS CURVE
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    range(1, len(loss_history) + 1),
    loss_history,
    marker="o"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Training cross-entropy"
)

plt.title(
    f"VQC Training Loss — "
    f"{n_qubits} Qubits — "
    f"Test Subject {test_subject}"
)

plt.grid(alpha=0.3)
plt.tight_layout()

loss_plot = (
    RESULTS_ROOT
    / f"loss_q{n_qubits}_subject_{test_subject}.png"
)

plt.savefig(
    loss_plot,
    dpi=300
)

plt.close()

# ============================================================
# CONFUSION MATRIX
# ============================================================

plt.figure(
    figsize=(7, 6)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=TASK_NAMES,
    yticklabels=TASK_NAMES
)

plt.xlabel(
    "Predicted task"
)

plt.ylabel(
    "True task"
)

plt.title(
    f"Confusion Matrix — "
    f"{n_qubits} Qubits — "
    f"Subject {test_subject}"
)

plt.tight_layout()

cm_plot = (
    RESULTS_ROOT
    / f"confusion_q{n_qubits}_subject_{test_subject}.png"
)

plt.savefig(
    cm_plot,
    dpi=300
)

plt.close()

# ============================================================
# NORMALISED OVERALL CONFUSION MATRICES
# ============================================================

for n_qubits in QUBIT_LIST:

    cm = confusion_matrices[n_qubits]

    row_sums = cm.sum(
        axis=1,
        keepdims=True
    )

    cm_normalised = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(
            cm,
            dtype=float
        ),
        where=row_sums != 0
    )

    plt.figure(
        figsize=(7, 6)
    )

    sns.heatmap(
        cm_normalised,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=TASK_NAMES,
        yticklabels=TASK_NAMES,
        vmin=0,
        vmax=1
    )

    plt.xlabel(
        "Predicted task"
    )

    plt.ylabel(
        "True task"
    )

    plt.title(
        f"Normalised Overall Confusion Matrix — "
        f"{n_qubits} Qubits"
    )

    plt.tight_layout()

    plt.savefig(
        RESULTS_ROOT /
        f"overall_confusion_normalised_q{n_qubits}.png",
        dpi=300
    )

    plt.close()

# ============================================================
# OVERALL TRAINING LOSS
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for n_qubits in QUBIT_LIST:

    histories = loss_histories[n_qubits]

    if len(histories) == 0:
        continue

    histories = np.asarray(
        histories
    )

    mean_loss = np.mean(
        histories,
        axis=0
    )

    std_loss = np.std(
        histories,
        axis=0
    )

    epochs = np.arange(
        1,
        len(mean_loss) + 1
    )

    plt.plot(
        epochs,
        mean_loss,
        marker="o",
        label=f"{n_qubits} qubits"
    )

    plt.fill_between(
        epochs,
        mean_loss - std_loss,
        mean_loss + std_loss,
        alpha=0.15
    )

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Training cross-entropy"
)

plt.title(
    "Overall VQC Training Loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "overall_training_loss.png",
    dpi=300
)

plt.close()

# ============================================================
# OVERALL TRAINING ACCURACY
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for n_qubits in QUBIT_LIST:

    histories = accuracy_histories[n_qubits]

    if len(histories) == 0:
        continue

    histories = np.asarray(
        histories
    )

    mean_accuracy = np.mean(
        histories,
        axis=0
    )

    std_accuracy = np.std(
        histories,
        axis=0
    )

    epochs = np.arange(
        1,
        len(mean_accuracy) + 1
    )

    plt.plot(
        epochs,
        mean_accuracy,
        marker="o",
        label=f"{n_qubits} qubits"
    )

    plt.fill_between(
        epochs,
        mean_accuracy - std_accuracy,
        mean_accuracy + std_accuracy,
        alpha=0.15
    )

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Training accuracy"
)

plt.title(
    "Overall VQC Training Accuracy"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "overall_training_accuracy.png",
    dpi=300
)

plt.close()

# ============================================================
# TASK F1 HEATMAP
# ============================================================

task_f1_summary = (
    task_results_df
    .groupby(
        ["task", "n_qubits"]
    )["f1"]
    .mean()
    .reset_index()
)

task_f1_pivot = (
    task_f1_summary
    .pivot(
        index="task",
        columns="n_qubits",
        values="f1"
    )
)

plt.figure(
    figsize=(9, 5)
)

sns.heatmap(
    task_f1_pivot,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    vmin=0,
    vmax=1
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Task"
)

plt.title(
    "Mean Task-wise F1 Across Quantum Dimensions"
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "task_f1_heatmap.png",
    dpi=300
)

plt.close()

# ============================================================
# OVERALL TRAINING SAMPLE DISTRIBUTION
# ============================================================

overall_task_counts = (
    task_counts_df[
        task_counts_df["n_qubits"] == 4
    ]
    .groupby("task")["count"]
    .mean()
    .reindex(TASK_NAMES)
)

plt.figure(
    figsize=(9, 6)
)

plt.bar(
    overall_task_counts.index,
    overall_task_counts.values
)

plt.xlabel(
    "Task"
)

plt.ylabel(
    "Mean number of training trials"
)

plt.title(
    "Training Trial Distribution Across Tasks"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "overall_training_task_distribution.png",
    dpi=300
)

plt.close()

# ============================================================
# PREDICTED CLASS DISTRIBUTION
# ============================================================

prediction_distribution = []

for n_qubits in QUBIT_LIST:

    for test_subject in SUBJECTS:

        result_path = (
            RESULTS_ROOT
            / f"qubits_{n_qubits}"
            / f"loso_test_{test_subject}.npz"
        )

        if not result_path.exists():
            continue

        result = np.load(
            result_path,
            allow_pickle=True
        )

        predictions = result[
            "predictions"
        ]

        for class_id, task_name in enumerate(TASK_NAMES):

            count = np.sum(
                predictions == class_id
            )

            prediction_distribution.append({

                "n_qubits":
                    n_qubits,

                "subject":
                    test_subject,

                "task":
                    task_name,

                "count":
                    count
            })

prediction_distribution_df = pd.DataFrame(
    prediction_distribution
)

prediction_summary = (
    prediction_distribution_df
    .groupby(
        ["n_qubits", "task"]
    )["count"]
    .mean()
    .reset_index()
)

plt.figure(
    figsize=(10, 6)
)

for task in TASK_NAMES:

    task_data = (
        prediction_summary[
            prediction_summary["task"] == task
        ]
    )

    plt.plot(
        task_data["n_qubits"],
        task_data["count"],
        marker="o",
        label=task
    )

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Mean number of predictions"
)

plt.title(
    "Predicted Task Distribution Across Quantum Dimensions"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "predicted_task_distribution.png",
    dpi=300
)

plt.close()

# ============================================================
# OVERALL CONFUSION MATRICES
# POOLED ACROSS ALL LOSO SUBJECTS
# ============================================================

for n_qubits in QUBIT_LIST:

    all_y_true = []
    all_y_pred = []

    for test_subject in SUBJECTS:

        result_path = (
            RESULTS_ROOT
            / f"qubits_{n_qubits}"
            / f"loso_test_{test_subject}.npz"
        )

        if not result_path.exists():

            print(
                f"Missing result: {result_path}"
            )

            continue

        data = np.load(
            result_path,
            allow_pickle=True
        )

        y_true = data["y_test"]
        y_pred = data["predictions"]

        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)

    # Convert to arrays

    all_y_true = np.asarray(
        all_y_true
    )

    all_y_pred = np.asarray(
        all_y_pred
    )

    # Overall confusion matrix

    overall_cm = confusion_matrix(
        all_y_true,
        all_y_pred,
        labels=[0, 1, 2, 3]
    )

    print("\n")
    print("=" * 70)
    print(
        f"OVERALL CONFUSION MATRIX — "
        f"{n_qubits} QUBITS"
    )
    print("=" * 70)

    print(overall_cm)

    # --------------------------------------------------------
    # PLOT
    # --------------------------------------------------------

    plt.figure(
        figsize=(7, 6)
    )

    sns.heatmap(
        overall_cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=TASK_NAMES,
        yticklabels=TASK_NAMES,
        cbar_kws={
            "label": "Number of test trials"
        }
    )

    plt.xlabel(
        "Predicted task"
    )

    plt.ylabel(
        "True task"
    )

    plt.title(
        f"Overall Confusion Matrix — "
        f"{n_qubits} Qubits"
    )

    plt.tight_layout()

    cm_plot = (
        RESULTS_ROOT
        / f"overall_confusion_q{n_qubits}.png"
    )

    plt.savefig(
        cm_plot,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved:",
        cm_plot
    )

# ============================================================
# COMPLETE
# ============================================================

print("\n")
print("=" * 80)
print("ALL VQC EXPERIMENTS COMPLETE")
print("=" * 80)

print(
    "Excel:",
    EXCEL_PATH
)

print(
    "Plots saved in:",
    RESULTS_ROOT
)



VQC EXPERIMENT: 4 QUBITS


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 002
--------------------------------------------------------------------------------
Training: (4369, 4)
Testing: (1484, 4)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...
Epoch   1/30 | Loss = 1.384400
Epoch   5/30 | Loss = 1.436020
Epoch  10/30 | Loss = 1.387816
Epoch  15/30 | Loss = 1.367980
Epoch  20/30 | Loss = 1.376047
Epoch  25/30 | Loss = 1.364009
Epoch  30/30 | Loss = 1.394266


Testing VQC...

Prediction distribution:
Class 0: 1 (0.07%)
Class 1: 1480 (99.73%)
Class 2: 0 (0.00%)
Class 3: 3 (0.20%)

Test accuracy: 0.2716
Balanced accuracy: 0.2487
Macro F1: 0.1077
Weighted F1: 0.1177

Confusion matrix:
[[  0 400   0   0]
 [  1 402   0   2]
 [  0 217   0   0]
 [  0 461   0   1]]
Saved: /home/master/MasterThesis/OPM-MEG MPS/results/vqc/experiment4/qubits_4/loso_test_002.npz


-------------------------------------------------